# Label choice in a care-management algorithm

## Section 5 — Rebuild and compare the ranking

We will train one LASSO model to predict `gagne_sum_t`, the recorded active chronic-condition count, then use the predicted values as another "risk score" to rank the patients and select the top 3% for the program.

We will then compare its top-3% allocation with the commercial algorithm's top 3% on the same held-out patients.

Following the commercial algorithm, race is excluded from model training. It is retained only to audit who each rule selects and how observed outcomes compare across groups.


## START/RESTART HERE

Before Task 1, run every setup code cell below in order. Each setup cell is
labeled `# RUN THIS CELL FIRST`; continue until you reach **Task 1**.

The setup imports packages, loads the data, fixes a 50/50 split and seed, and
defines the short classroom penalty grid. The full nine-value reference grid
remains visible as a comment for transparency but is not run in class.


In [1]:
# RUN THIS CELL FIRST
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
# RUN THIS CELL FIRST
DATA_COMMIT = "daceb25bba00e65d7b05882f049e229a8bedb60c"
DATA_URL = (
    "https://gitlab.com/labsysmed/dissecting-bias/-/raw/"
    f"{DATA_COMMIT}/data/data_new.csv"
)
LOCAL_DATA_PATH = Path("data_new.csv")


def load_data(required_columns):
    """Load the course data from a local copy or the pinned public URL."""
    source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.is_file() else DATA_URL
    data = pd.read_csv(source)

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    return data


In [3]:
# RUN THIS CELL FIRST
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [4]:
# RUN THIS CELL FIRST
RANDOM_STATE = 707
TEST_SIZE = 0.5

# Full reference grid used during instructor validation:
# RELATIVE_ALPHA_GRID = np.logspace(-4, 0, 9)

# The active classroom grid keeps the validated full-grid winner and its
# immediate neighboring candidates while reducing the search to 15 fold-fits.
RELATIVE_ALPHA_GRID = np.array([1e-3, 10**-2.5, 1e-2])

required_columns = ["race", "risk_score_t", "gagne_sum_t"]
df = load_data(required_columns)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


## Task 1 — Build leakage-safe modeling data

### 1A — Choose the predictors

Create `feature_columns`, a Python list in dataset order containing:

- every prior-year variable whose name ends in `_tm1`; and
- every demographic variable whose name starts with `dem_` and does not
  contain `race`.

Display the length of the list. The verified data should produce 149 predictor
names.


In [5]:
feature_columns = [
    column
    for column in df.columns
    if (
        column.endswith("_tm1")
        or (column.startswith("dem_") and "race" not in column)
    )
]

len(feature_columns)


149

### 1B — Make the train-test split

Use one `train_test_split` call on the source-row index with
`test_size=TEST_SIZE` and `random_state=RANDOM_STATE`. Store the resulting NumPy
arrays as `train_index` and `test_index`.

Then create:

- `X_train` and `X_test`: predictor DataFrames using `feature_columns`; and
- `y_train` and `y_test`: matching `gagne_sum_t` Series.

Display the four shapes. The split should produce 24,392 rows in each half,
with 149 predictor columns.


In [6]:
train_index, test_index = train_test_split(
    df.index.to_numpy(),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train = df.loc[train_index, feature_columns]
X_test = df.loc[test_index, feature_columns]
y_train = df.loc[train_index, "gagne_sum_t"]
y_test = df.loc[test_index, "gagne_sum_t"]

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((24392, 149), (24392, 149), (24392,), (24392,))

### 1C — Check the feature names

Print two direct Boolean checks that confirm:

1. no name in `feature_columns` contains `race`; and
2. no name in `feature_columns` ends in current-year `_t`.

Both checks should display `True`.


In [7]:
print(
    "No race feature:",
    not any("race" in column for column in feature_columns),
)
print(
    "No current-year _t feature:",
    not any(column.endswith("_t") for column in feature_columns),
)


No race feature: True
No current-year _t feature: True


## Task 2 — Train and evaluate one illness-burden LASSO

### 2A — Train LASSO

**Goal:** Train a LASSO model that predicts each patient's recorded active
chronic-condition count (`gagne_sum_t`) from the 149 predictors. Use 5-fold
cross-validation on the training half to choose the penalty strength `alpha`.

**Your task:** Complete the code cell below.

Then fit `illness_search` on `X_train` and `y_train`.

**Notes:**

- The grid has only three values to keep the run short in class. The instructor
  has already checked that the best value from a larger search is among them.
- Fitting runs 15 fits (3 `alpha` values × 5 folds), then refits the best
  model on the whole training half. Expect it to take a little while.

In [8]:
alpha_grid = RELATIVE_ALPHA_GRID * y_train.std(ddof=0)  # USE THIS ALPHA GRID IN CLASS

lasso_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "model",
            Lasso(max_iter=200_000, tol=0.001),
        ),
    ]
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

illness_search = GridSearchCV(
    lasso_pipeline,
    param_grid={"model__alpha": alpha_grid},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
)
illness_search.fit(X_train, y_train);


### 2B — Evaluate on the untouched test half

**Goal:** Check whether the LASSO predicts condition counts better than a
simple baseline on patients it has never seen. The baseline predicts the same
number for everyone: the average `gagne_sum_t` in the **training** half.

**Your task:** In the empty code cell below, create:

| Object | What it should be |
|---|---|
| `illness_predictions` | The fitted `illness_search`'s predictions for `X_test` (a NumPy array with 24,392 values) |
| `mean_baseline_predictions` | A NumPy array with 24,392 values, each equal to the mean of `y_train` |
| `baseline_rmse` | The RMSE of `mean_baseline_predictions` against `y_test` |
| `lasso_rmse` | The RMSE of `illness_predictions` against `y_test` |

Use the `rmse(y_true, y_pred)` helper defined in the setup cells. Print the two
RMSE values rounded to three decimal places. There is no need to report the
selected `alpha` or to build a DataFrame for the results.

**Check:** A lower RMSE means more accurate predictions. If the LASSO has
learned anything useful, `lasso_rmse` should be smaller than `baseline_rmse`.


In [9]:
illness_predictions = illness_search.predict(X_test)
mean_baseline_predictions = np.repeat(y_train.mean(), len(y_test))

baseline_rmse = rmse(y_test, mean_baseline_predictions)
lasso_rmse = rmse(y_test, illness_predictions)

print(f"Baseline test RMSE: {baseline_rmse:.3f}")
print(f"LASSO test RMSE: {lasso_rmse:.3f}")


Baseline test RMSE: 1.939
LASSO test RMSE: 1.015


**Solution — interpretation of Task 2B:** Held-out RMSE falls from `1.939` for the
training-mean baseline to `1.015` for LASSO, so the LASSO-based algorithm
predicts the recorded condition-count proxy more accurately. Both errors are
measured in recorded active chronic-condition-count units.


## Task 3 — Compare two fixed-capacity decisions with a random benchmark

**Goal:** Compare three ways of choosing which test-half patients enter the
care-management program:

1. **LASSO policy:** the top 3% by predicted condition count from Task 2;
2. **Commercial policy:** the top 3% by the commercial score `risk_score_t`;
   and
3. **Random policy:** 3% chosen uniformly at random, as a benchmark.

All three policies choose from the same 24,392 test-half rows and have the same capacity: 3% of 24,392, rounded up, is **732** places.

**Your task:**
Read the code cell below and understand the logic.

Then, run the code cell below without editing it. It defines
`select_exact_top_fraction(scores, fraction)`, which applies the same top-3%
rule as Section 3. It returns a Boolean Series that is `True` for the selected
rows, with the highest scores selected first.


In [10]:
def select_exact_top_fraction(scores, fraction=0.03):
    """Select the highest-scoring fraction of rows.

    Parameters
    ----------
    scores : pandas.Series
        Scores indexed by source-row identifier. Larger values receive higher
        priority.
    fraction : float, default=0.03
        Fraction of rows to select.

    Returns
    -------
    pandas.Series
        Boolean Series aligned with ``scores``, where ``True`` identifies a
        selected row.

    Notes
    -----
    The function selects ``ceil(fraction * n)`` rows. Ties at the selection
    boundary are resolved by ascending source-row index.
    """
    number_selected = int(np.ceil(fraction * len(scores)))
    ranking = pd.DataFrame(
        {"score": scores, "source_row": scores.index},
        index=scores.index,
    )
    selected_index = (
        ranking.sort_values(
            ["score", "source_row"],
            ascending=[False, True],
            kind="mergesort",
        )
        .head(number_selected)
        .index
    )

    selected = pd.Series(False, index=scores.index, name="selected")
    selected.loc[selected_index] = True
    return selected


### 3A — Apply the LASSO-based policy

**Goal:** Build one table of the test-half patients, then mark the 732 patients the LASSO policy would select.

**Your task:** Complete the code cell below. It creates `test_results`, which
has one row per test-half patient, uses the original row labels of `df` as its
index, and starts with the columns `race`, `risk_score_t`, and `gagne_sum_t`.
Add two columns:

- `lasso_prediction`: the values in `illness_predictions`, turned into a
  Series. Give it the **same index as `test_results`** so that each prediction
  lines up with the right patient. (`illness_predictions` is a plain NumPy
  array, so it has no row labels of its own.)
- `lasso_selected`: the result of applying `select_exact_top_fraction` to the
  `lasso_prediction` column with `fraction=0.03`.

**Check:** The printed number of LASSO-selected rows should be 732.


In [11]:
test_results = df.loc[
    test_index,
    ["race", "risk_score_t", "gagne_sum_t"],
].copy()

test_results["lasso_prediction"] = pd.Series(
    illness_predictions,
    index=test_results.index,
)

test_results["lasso_selected"] = select_exact_top_fraction(
    test_results["lasso_prediction"],
    fraction=0.03,
)

print("LASSO-selected rows:", test_results["lasso_selected"].sum())


LASSO-selected rows: 732


### 3B — Apply the commercial-score policy

**Goal:** Mark the 732 test-half patients the commercial algorithm would
select, using the same top-3% rule as 3A.

**Your task:** Complete the code cell below. Add the column
`commercial_selected` to `test_results` by applying `select_exact_top_fraction` to the `risk_score_t` column with `fraction=0.03`.

**Check:** The printed number of commercial-score-selected rows should also be 732.


In [12]:
test_results["commercial_selected"] = select_exact_top_fraction(
    test_results["risk_score_t"],
    fraction=0.03,
)

print(
    "Commercial-score-selected rows:",
    test_results["commercial_selected"].sum(),
)


Commercial-score-selected rows: 732


### 3C — Count treated people in each group

**Goal:** Find out how many Black and White patients each policy selects. Both
policies have the same 732 places, so any difference shows how each one
divides those places between the two groups.

**Your task:** Complete the code cell below to create:

| Object | What it should be |
|---|---|
| `lasso_treated` | The rows of `test_results` where `lasso_selected` is `True` |
| `commercial_treated` | The rows of `test_results` where `commercial_selected` is `True` |
| `lasso_treated_counts` | The number of Black and White patients in `lasso_treated`, as a Series |
| `commercial_treated_counts` | The same counts for `commercial_treated` |

Reindex both count Series with `race_order` so that Black is listed first and
White second. The cell prints the two Series directly, so there is no need to
build a DataFrame for the results.

**Check:** Within each policy, the Black and White counts should add up to
732.


In [13]:
race_order = ["black", "white"]

lasso_treated = test_results.loc[test_results["lasso_selected"]]
commercial_treated = test_results.loc[
    test_results["commercial_selected"]
]

lasso_treated_counts = (
    lasso_treated["race"]
    .value_counts()
    .reindex(race_order, fill_value=0)
)
commercial_treated_counts = (
    commercial_treated["race"]
    .value_counts()
    .reindex(race_order, fill_value=0)
)

print("LASSO policy — treated people by race")
print(lasso_treated_counts)
print("\nCommercial policy — treated people by race")
print(commercial_treated_counts)


LASSO policy — treated people by race
race
black    204
white    528
Name: count, dtype: int64

Commercial policy — treated people by race
race
black    139
white    593
Name: count, dtype: int64


### 3D — Compare observed illness burden across policies

**Goal:** For each policy, compare how sick the selected Black and White
patients are, measured by their actual recorded condition count
(`gagne_sum_t`). A random policy is included as a benchmark: a policy that
targets sicker patients should select patients with higher condition counts
than random selection does.

**Your task:** Complete the code cell below to create:

| Object | What it should be |
|---|---|
| `number_treated` | The capacity, 732 (already filled in) |
| `random_treated` | `number_treated` rows sampled at random from `test_results`, using `random_state=RANDOM_STATE` so everyone gets the same sample |
| `random_mean_gagne_by_race` | The mean `gagne_sum_t` among Black and among White patients in `random_treated` |
| `lasso_mean_gagne_by_race` | The same means for `lasso_treated` |
| `commercial_mean_gagne_by_race` | The same means for `commercial_treated` |
| `random_gagne_gap`, `lasso_gagne_gap`, `commercial_gagne_gap` | For each policy, the Black mean **minus** the White mean |

A positive gap means the selected Black patients have more recorded conditions, on average, than the selected White patients.

**Question to consider:** Compared with random selection, does the commercial
policy select patients with more recorded conditions?


In [14]:
number_treated = int(np.ceil(0.03 * len(test_results)))
random_treated = test_results.sample(
    n=number_treated,
    random_state=RANDOM_STATE,
)

random_mean_gagne_by_race = (
    random_treated.groupby("race")["gagne_sum_t"]
    .mean()
    .reindex(race_order)
)

lasso_mean_gagne_by_race = (
    lasso_treated.groupby("race")["gagne_sum_t"]
    .mean()
    .reindex(race_order)
)
commercial_mean_gagne_by_race = (
    commercial_treated.groupby("race")["gagne_sum_t"]
    .mean()
    .reindex(race_order)
)

random_gagne_gap = (
    random_mean_gagne_by_race["black"]
    - random_mean_gagne_by_race["white"]
)
lasso_gagne_gap = (
    lasso_mean_gagne_by_race["black"]
    - lasso_mean_gagne_by_race["white"]
)
commercial_gagne_gap = (
    commercial_mean_gagne_by_race["black"]
    - commercial_mean_gagne_by_race["white"]
)

print("Random policy — mean gagne_sum_t by race")
print(random_mean_gagne_by_race.round(3))
print(f"Black - White gap: {random_gagne_gap:.3f}")

print("\nCommercial policy — mean gagne_sum_t by race")
print(commercial_mean_gagne_by_race.round(3))
print(f"Black - White gap: {commercial_gagne_gap:.3f}")

print("\nLASSO policy — mean gagne_sum_t by race")
print(lasso_mean_gagne_by_race.round(3))
print(f"Black - White gap: {lasso_gagne_gap:.3f}")


Random policy — mean gagne_sum_t by race
race
black    1.876
white    1.277
Name: gagne_sum_t, dtype: float64
Black - White gap: 0.599

Commercial policy — mean gagne_sum_t by race
race
black    6.712
white    5.292
Name: gagne_sum_t, dtype: float64
Black - White gap: 1.420

LASSO policy — mean gagne_sum_t by race
race
black    7.279
white    7.477
Name: gagne_sum_t, dtype: float64
Black - White gap: -0.198


**Questions:**

Use the outputs from Tasks 3C and 3D to answer:

1. Does the commercial-score policy select patients with higher observed `gagne_sum_t` than uniform random selection within both racial groups? What does the result suggest?
2. What are the differences between the commercial-score policy and the LASSO-based policy? Which policy seems to be fairer?


**Answer:** [Write your answers here.]


**Solution — answers to Task 3D and the final questions:**

1. **Commercial policy compared with random selection.** Yes, in both groups.
   The commercial policy selects patients with mean recorded `gagne_sum_t` of
   `6.712` among Black patients and `5.292` among White patients. Uniform random
   selection gives means of `1.876` and `1.277`, respectively. The commercial
   score therefore helps target patients with higher recorded illness burden
   than uniform random selection, even though predicting illness burden is
   not the commercial algorithm's training objective. Better targeting than
   random selection does not by itself establish fairness.

2. **Commercial policy compared with the LASSO policy.** Both select exactly
   732 patients. The LASSO policy selects 204 Black and 528 White patients,
   compared with 139 Black and 593 White patients for the commercial policy:
   65 more Black patients and 65 fewer White patients at the same capacity.
   Mean observed `gagne_sum_t` under LASSO is `7.279` for selected Black
   patients and `7.477` for selected White patients, higher than the
   commercial-policy means in both groups. The Black-minus-White gap is
   `-0.198` under LASSO versus `+1.420` under the commercial policy.

   If the fairness criterion is selecting patients at comparable levels of
   recorded illness burden across racial groups, LASSO appears fairer here:
   its absolute selected-group gap is much smaller, although its direction
   reverses in this test sample. Changing the prediction label to recorded
   illness burden also assigns more of the fixed places to Black patients.
   These descriptive comparisons do not measure who benefits most from the
   program; recorded condition counts are a proxy for need, so they do not
   establish fairness under every possible criterion.
